In [1]:
from playpen import EpisodeBuffer
%load_ext autoreload
%autoreload 2

This notebook demonstrates how to use the clemcore OpenEnv integration to run a
single-player game with the game **"Wordle"**, and how to plug in a self-hosted
OpenAI-compatible model server as an agent backend.

You will learn how to:

1. Set up the environment and install dependencies.
2. Register and configure a model as an agent backend.
3. Implement a simple custom describer agent.
4. Run an automated gameplay session and inspect the interactions.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/phisad/playpen/blob/main/examples/openenv/wordle.ipynb
)

# 1. Environment setup and dependency installation

## 1.1. Setup environment

We start by specifying:

- The game name (here we use `"wordle"`, but this pattern works for any 1‑player game).
- `CLEMBENCH_HOME`, the local path where the `clembench` repository is located.
  This environment variable is used by the CLI and Python APIs to locate games.

In [2]:
import os

# Specify the game name here (this code can be adapted to any 1-player game)
GAME_NAME = "wordle"

# Local clone location of the clembench repository
CLEMBENCH_HOME = os.path.expanduser("~/git/clembench")

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

## 1.2. Install games and dependencies

In this step we:

1. Clone the `clembench` repository (skip if you already have it).
2. Install its Python dependencies into the current Jupyter kernel.
3. Verify that `clembench` is installed and that the game is visible.

You should:

- See a valid version printed by `clem --version`.
- See `wordle` listed by `clem list games -s wordle`.

If you do **not** see `wordle`, double‑check `CLEMBENCH_HOME` and that the clone succeeded.

In [ ]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
%pip install --upgrade ipywidgets jupyter_client

In [ ]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s $GAME_NAME

# 2. Implement a TRL guesser agent

Next, we implement a TRL-based guesser agent by subclassing `ClemAgent`.
The agent:

- Uses the `trl.GRPOTrainer` to generate the completions.
- Generates a guess based on the full interaction history in the Wordle game.


# 4. Run an automated gameplay session

We now:

1. Set up an OpenEnv environment for the game where we use the registered `"clp-chat"` model to play `player_1` as part of the environment.
2. Run through a single episode and log the interaction at each step where we use our custom describer to play `player_0` as the learning agent.

Note that:
- Setting `--single_pass` allows to iterate through all game instances only once times, e.g., for evaluation.
- You can also restrict which instances are used via the `--split [train,validation]` option.

## 4.1. Starting the server and connecting the client

Open a terminal in the notebook folder and run the `clem serve` command to start the OpenEnv environment:

```bash
clem serve --game wordle
```

The output should look similar to:
```
INFO:     Started server process [73210]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

From this log you obtain the host and port (here `http://0.0.0.0:8000`) that the client should connect to.

In [3]:
from clemcore.clemgame import ClemGameEnv

game_env = ClemGameEnv(base_url="http://0.0.0.0:8000")

## 4.2. Run through an episode and log the interaction

In [ ]:
from dataclasses import dataclass, field
from playpen.agents.openenv import ClemGameEnvAgent


@dataclass
class GrpoEpisodeRollout:
    """Collect training info about a single episode.
    
    Following TRL's pattern, we accumulate all turns into a single completion
    sequence, using env_masks to distinguish model tokens (1) from env tokens (0).
    """
    prompt_ids: list[int] = field(default_factory=list)
    completion_ids: list[int] = field(default_factory=list)
    logprobs: list[float] = field(default_factory=list)
    env_masks: list[int] = field(default_factory=list)

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_masks.clear()


@dataclass
class GrpoEpisodeRollouts:
    """Collect training info about all episodes for a batch."""
    prompt_ids: list[list[int]] = field(default_factory=list)
    completion_ids: list[list[int]] = field(default_factory=list)
    logprobs: list[list[float]] = field(default_factory=list)
    env_mask: list[list[int]] = field(default_factory=list)

    def append(self, rollout: GrpoEpisodeRollout):
        """Append a completed episode rollout."""
        self.prompt_ids.append(list(rollout.prompt_ids))
        self.completion_ids.append(list(rollout.completion_ids))
        self.logprobs.append(list(rollout.logprobs))
        self.env_mask.append(list(rollout.env_masks))

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_mask.clear()

In [ ]:
import trl
from playpen.agents import ClemObservation, ClemAgent
from trl.experimental.openenv import generate_rollout_completions


class WordleAgent(ClemAgent):
    """Agent that plays Wordle using TRL's GRPOTrainer for generation.
    
    Handles all tokenization and env_mask logic internally, accumulating
    the full episode trajectory for GRPO training.

    This implementation is based on the wordle openenv example given in the trl repository at
    https://github.com/huggingface/trl/blob/v0.28.0/examples/scripts/openenv/wordle.py
    """

    def __init__(self, trainer: trl.GRPOTrainer):
        super().__init__()
        self.trainer = trainer
        self.tokenizer = trainer.processing_class
        self.episode = GrpoEpisodeRollout()
        self._first_turn = True

    def act(self, last: ClemObservation) -> str:
        # On first turn, set prompt_ids from initial observation
        if self._first_turn:
            prompt_text = self.tokenizer.apply_chat_template(
                self.history,
                add_generation_prompt=True,
                tokenize=False,
            )
            self.episode.prompt_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
            self._first_turn = False
        else:
            # Not first turn: the observation content is env feedback from previous action
            env_feedback_ids = self.tokenizer.encode(last.content, add_special_tokens=False)
            self.episode.completion_ids.extend(env_feedback_ids)
            self.episode.logprobs.extend([0.0] * len(env_feedback_ids))
            self.episode.env_masks.extend([0] * len(env_feedback_ids))

        # Build full prompt with conversation history for generation
        prompt_text = self.tokenizer.apply_chat_template(
            self.history,
            add_generation_prompt=True,
            tokenize=False,
        )

        # Generate using vLLM via TRL's rollout helper
        outputs = generate_rollout_completions(self.trainer, [prompt_text])[0]

        # Add model-generated tokens (mask=1)
        self.episode.completion_ids.extend(outputs["completion_ids"])
        self.episode.logprobs.extend(outputs["logprobs"])
        self.episode.env_masks.extend([1] * len(outputs["completion_ids"]))

        # Decode and return text action
        return outputs.get("text") or self.tokenizer.decode(
            outputs["completion_ids"], skip_special_tokens=True
        )

    def get_episode(self) -> "GrpoEpisodeRollout":
        """Return the accumulated episode rollout for GRPO training."""
        return self.episode

    def reset(self):
        """Reset agent state for a new episode."""
        super().reset()
        self.episode.reset()
        self._first_turn = True

In [ ]:
from dataclasses import asdict


def rollout_episode(env: ClemGameEnv, agent: ClemGameEnvAgent) -> GrpoEpisodeRollout:
    """Play a single episode of Wordle and collect GRPO training data.
    
    The agent handles all tokenization and env_mask logic internally.
    
    Returns:
        GrpoEpisodeRollout with accumulated prompt_ids, completion_ids, logprobs, and env_masks.
    """
    obs = env.reset()
    while not obs.done:
        action = agent(obs)
        obs = env.step(action)
    return agent.wrapped_agent.get_episode()


def rollout_func(prompts: list[str], trainer: trl.GRPOTrainer) -> dict:
    """Custom rollout function for TRL GRPOTrainer with OpenEnv.
    
    Note:
        rollout_func receives prompts already duplicated K times (num_generations).
        For example, with batch size 4 and num_generations 8, there are 32 prompts.
        Since Wordle always starts with the same initial state, we ignore the prompt
        content and just run that many episodes.
    """
    # Create the TRL wordle agent and apply the openenv wrapper for convenience
    agent = ClemGameEnvAgent(WordleAgent(trainer))
    rollouts = GrpoEpisodeRollouts()
    try:
        # We only have a single env started, so we must go sequentially here
        for _ in prompts:
            rollout = rollout_episode(game_env, agent)
            rollouts.append(rollout)
            agent.reset()  # Reset for next episode
        return asdict(rollouts)
    finally:
        rollouts.reset()
        agent.reset()

In [ ]:
# TRL supports rollout_func only for vllm mode
# TRL has problems with PEFT + vllm in colocate mode, but requires 2 GPUs to run on server mode: one for inference and one for training (which makes training slower; and misses kind of the point of GRPO not having a separate model loaded)
# hence, TRL full training with colocate mode and
from peft import LoraConfig
from transformers import BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3-8B-chat-hf"
grpo_config = trl.GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.6,
    num_generations=4,  # Default
    per_device_train_batch_size=4,  # Default
    num_train_epochs=10,
    disable_dropout=True,
    max_prompt_length=None,
    max_completion_length=2048,  # Full episode length
    output_dir=f"models/grpo/wordle/{MODEL_ID}"
)
peft_config = LoraConfig(  # see https://huggingface.co/docs/trl/sft_trainer#training-adapters
    r=16, lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_token"],
    task_type="CAUSAL_LM",
)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)
# Initialize trainer context
grpo_trainer = trl.GRPOTrainer(
    model=MODEL_ID,
    rollout_func=rollout_func,  # repeats each game instance in batch K times (using RepeatSampler)
    train_dataset=...,
    args=grpo_config,
    peft_config=peft_config,
    model_init_kwargs={"quantization_config": bnb_config}
)

# Train on the dataset; this will save only the adapters to the checkpoints directory
grpo_trainer.train()